# Text Preprocessing: Full Pipeline

Putting it all together: text cleaning -> case normalization ->
tokenization -> stop word removal -> stemming/lemmatization.

The order matters - cleaning and case normalization should generally
happen *before* tokenization, and stop word removal *before*
stemming (no point stemming words you're about to throw away).

In [ ]:
import re
import nltk

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()


def preprocess_text(text, verbose=False):
    # 1. Case normalization
    text = text.lower()

    # 2. Text cleaning
    text = re.sub(r'<[^>]+>', '', text)          # HTML tags
    text = re.sub(r'http\S+|www\S+', '', text)   # URLs
    text = re.sub(r'[^a-z0-9\s]', '', text)       # punctuation/symbols
    text = re.sub(r'\s+', ' ', text).strip()      # extra whitespace
    if verbose:
        print('after cleaning:  ', text)

    # 3. Tokenization
    tokens = word_tokenize(text)
    if verbose:
        print('after tokenizing:', tokens)

    # 4. Stop word removal
    tokens = [t for t in tokens if t not in stop_words]
    if verbose:
        print('after stopwords: ', tokens)

    # 5. Stemming
    tokens = [stemmer.stem(t) for t in tokens]
    if verbose:
        print('after stemming:  ', tokens)

    return tokens


sample = "The CEO announced a synergistic paradigm shift to leverage blockchain technology for optimizing ROI metrics."
result = preprocess_text(sample, verbose=True)
print('\nfinal tokens:', result)

## Why this matters: sentiment analysis vocabulary

Without preprocessing, `"Great movie!"`, `"great movie"`, and
`"GREAT MOVIE!!!"` look like three unrelated training examples. After
preprocessing they collapse into one consistent pattern.

In [ ]:
raw_examples = ["Great movie!", "great movie", "GREAT MOVIE!!!"]

raw_vocab = set()
for ex in raw_examples:
    raw_vocab.update(ex.split())

processed_vocab = set()
for ex in raw_examples:
    processed_vocab.update(preprocess_text(ex))

print("raw vocabulary:      ", raw_vocab, '(size', len(raw_vocab), ')')
print("processed vocabulary:", processed_vocab, '(size', len(processed_vocab), ')')

## Why this matters: search matching

A query and a document can mean the same thing without sharing any
exact words. Stemming/lemmatization aim to normalize related words
(`"running"`, `"runners"`) to a shared root so a match can be found
even when the exact word forms differ.

In [ ]:
query = "running shoes"
document = "best shoes for runners"

query_tokens = set(preprocess_text(query))
doc_tokens = set(preprocess_text(document))

print("query tokens:   ", query_tokens)
print("document tokens:", doc_tokens)
print("overlap:        ", query_tokens & doc_tokens)

**Note:** `"shoe"` matches directly. `"run"` and `"runner"` *don't*
fully collapse here - the Porter stemmer turns `"running"` into
`"run"` but `"runners"` into `"runner"`, not `"run"`. This is a good,
real example of stemming being **fast but crude** (see
`05_stemming_lemmatization.ipynb`): it applies suffix-stripping rules
without a dictionary, so related words don't always land on exactly
the same root. A lemmatizer or a smarter stemmer would close this gap.

## Best practices
1. Understand your data before preprocessing
2. Preprocess for the task at hand - not every task needs every step
3. Preserve important information - don't over-clean
4. Validate results manually, don't just trust the pipeline
5. Document the pipeline and keep train/test preprocessing identical

## Common pitfalls
1. Over-preprocessing - stripping out signal the model needed
2. Inconsistent preprocessing between training and test/production data
3. Ignoring domain specifics (medical, legal, and social-media text all
   need different treatment)
4. Not handling edge cases (empty strings, non-English text, emojis)
5. Inefficient preprocessing code that becomes a bottleneck on large datasets